# Part 25b: Scikit-learn Core -- Validation and Modern Features

**Dataset:** Hotel Booking Demand (same as Part 25a)

**Covers sections 7-9:** cross-validation, learning curves, model persistence. Plus Section 10: modern sklearn features (pandas output, feature names, threshold tuning).

Run the collapsed **Setup** section before any other cell.


## 0. Setup

Re-runs imports, data loading, and model fitting from Part 25a so this notebook is self-contained.

In [ ]:
from __future__ import annotations

# stdlib
from pathlib import Path
from typing import Any

# tables
from great_tables import GT, md as gt_md

# persistence
import joblib

# visualisation
from lets_plot import (
    LetsPlot,
    aes,
    as_discrete,
    geom_bar,
    geom_hline,
    geom_line,
    geom_point,
    geom_ribbon,
    ggplot,
    labs,
    scale_color_manual,
    scale_fill_manual,
)

# logging
from loguru import logger

# data
import numpy as np
import pandas as pd

# sklearn -- models
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

# sklearn -- metrics
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# sklearn -- model selection
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline

# sklearn -- transformers
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# project branding
from ark.plot.gt_style import metrics_report, themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, BRAND_PALETTE, DANGER, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html()

In [ ]:
DATA_URL: str = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
DATA_PATH: Path = Path("data/hotel_bookings.csv")
MODEL_DIR: Path = Path("models")

In [ ]:
def load_hotel_data(
    path: Path = DATA_PATH,
    url: str = DATA_URL,
) -> pd.DataFrame:
    """Load hotel bookings, downloading from url on first call."""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        raw = pd.read_csv(url)
        raw.to_csv(path, index=False)
        logger.success(f"Downloaded {len(raw):,} rows to {path}")
    else:
        raw = pd.read_csv(path)
        logger.info(f"Loaded {len(raw):,} rows from cache: {path}")
    df = raw[(raw["adr"] > 0) & (raw["adr"] <= 1000)].reset_index(drop=True)
    logger.info(f"After cleaning: {len(df):,} rows")
    return df

In [ ]:
df: pd.DataFrame = load_hotel_data()

In [ ]:
TARGET_REG: str = "adr"  # Average Daily Rate in euros (continuous)
TARGET_CLF: str = "is_canceled"  # 0 = kept, 1 = cancelled (binary)

In [ ]:
FEATURES: list[str] = [
    "lead_time",  # days booked in advance
    "stays_in_weekend_nights",  # weekend nights stayed
    "stays_in_week_nights",  # weekday nights stayed
    "adults",  # number of adults
    "previous_cancellations",  # prior cancellations by this guest
    "booking_changes",  # post-booking modifications
]

In [ ]:
X: np.ndarray = df[FEATURES].to_numpy(dtype=float)
y_reg: np.ndarray = df[TARGET_REG].to_numpy(dtype=float)
y_clf: np.ndarray = df[TARGET_CLF].to_numpy(dtype=int)
logger.info(f"X: {X.shape}  y_reg: {y_reg.shape}  y_clf: {y_clf.shape}")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y_reg, test_size=0.2, random_state=42)
logger.info(f"Train: {X_tr.shape}  Test: {X_te.shape}")

In [ ]:
scaler: StandardScaler = StandardScaler()
scaler.fit(X_tr)
logger.info(f"Learned means:  {scaler.mean_.round(2)}")
logger.info(f"Learned stds:   {scaler.scale_.round(2)}")

In [ ]:
X_train_s: np.ndarray = scaler.transform(X_tr)
X_test_s: np.ndarray = scaler.transform(X_te)
for i, feat in enumerate(FEATURES):
    logger.info(f"  {feat:<30} min={X_train_s[:, i].min():>6.2f}  max={X_train_s[:, i].max():>6.2f}")

In [ ]:
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
sc_clf: StandardScaler = StandardScaler()
Xc_train_s: np.ndarray = sc_clf.fit_transform(Xc_tr)
Xc_test_s: np.ndarray = sc_clf.transform(Xc_te)
logger.info(f"Train cancel rate: {yc_tr.mean():.2%}  Test cancel rate: {yc_te.mean():.2%}")

In [ ]:
lin_reg: LinearRegression = LinearRegression()
lin_reg.fit(X_train_s, y_tr)
logger.success("LinearRegression fitted")

In [ ]:
log_reg: LogisticRegression = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)
log_reg.fit(Xc_train_s, yc_tr)
logger.success("LogisticRegression fitted")

## 7. Cross-Validation

**From Part 24 (ML Workflow, Section 5):** a single train/test split gives a noisy, variance-prone estimate of generalisation. Cross-validation rotates the held-out subset across the entire training set.

<div style="font-family:monospace;background:#F8FAFC;border:1px solid #E2E8F0;border-radius:8px;padding:14px 18px;margin:12px 0;font-size:13px;line-height:1.9">
Fold 1: [<b>val</b>][train][train][train][train] &rarr; score<sub>1</sub><br>
Fold 2: [train][<b>val</b>][train][train][train] &rarr; score<sub>2</sub><br>
Fold 3: [train][train][<b>val</b>][train][train] &rarr; score<sub>3</sub><br>
Fold 4: [train][train][train][<b>val</b>][train] &rarr; score<sub>4</sub><br>
Fold 5: [train][train][train][train][<b>val</b>] &rarr; score<sub>5</sub><br>
Final estimate = mean(score<sub>1..5</sub>) +/- std
</div>

Each fold acts as a validation set exactly once. The final estimate is `mean +/- std` across all folds.

### 7.1 Define the regression pipeline for CV

We wrap scaler + model in a `Pipeline` so the scaler is re-fit on each fold's training subset automatically -- no leakage.

In [ ]:
reg_pipe: Pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

### 7.2 Run KFold cross-validation for regression

In [ ]:
kf: KFold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_reg: np.ndarray = cross_val_score(
    reg_pipe,
    X_tr,
    y_tr,
    cv=kf,
    scoring="neg_mean_absolute_error",
)
logger.info(f"CV MAE per fold: {(-cv_reg).round(2)}")
logger.info(f"Mean +/- Std: {-cv_reg.mean():.2f} +/- {cv_reg.std():.2f}")

### 7.3 Define the classification pipeline for CV

In [ ]:
clf_pipe: Pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]
)

### 7.4 Run StratifiedKFold cross-validation for classification

`StratifiedKFold` ensures the class ratio is preserved in every fold.

In [ ]:
skf: StratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_clf: np.ndarray = cross_val_score(
    clf_pipe,
    Xc_tr,
    yc_tr,
    cv=skf,
    scoring="f1",
)
logger.info(f"CV F1 per fold: {cv_clf.round(3)}")
logger.info(f"Mean +/- Std: {cv_clf.mean():.3f} +/- {cv_clf.std():.3f}")

::: {.callout-note icon=false}
## <i class="bi bi-info-circle-fill" style="color:#0369A1"></i>&nbsp; KFold vs StratifiedKFold

**KFold** -- use for regression (continuous targets).

**StratifiedKFold** -- use for classification. Preserves the class ratio in every fold, preventing a fold from being all one class on heavily imbalanced datasets.
:::

## 8. Learning Curves

**From Part 24 (ML Workflow, Section 6 -- Bias-Variance tradeoff):** a learning curve plots train and validation score as training set size grows.

- **Train >> validation, gap shrinks**: high variance (overfitting) -- gather more data or add regularisation.
- **Both scores high error, flat**: high bias (underfitting) -- add features or try a more expressive model.

### 8.1 Compute the learning curve

`learning_curve` re-trains the pipeline at each specified size using CV for the validation score.

In [ ]:
train_sizes: np.ndarray
train_lc_scores: np.ndarray
val_lc_scores: np.ndarray
train_sizes, train_lc_scores, val_lc_scores = learning_curve(
    reg_pipe,
    X_tr,
    y_tr,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
logger.info(f"Training sizes used: {train_sizes}")

### 8.2 Plot the learning curve

In [ ]:
lc_df: pd.DataFrame = pd.DataFrame(
    {
        "n": np.tile(train_sizes, 2),
        "score": np.concatenate([-train_lc_scores.mean(1), -val_lc_scores.mean(1)]),
        "err": np.concatenate([train_lc_scores.std(1), val_lc_scores.std(1)]),
        "split": ["Train"] * len(train_sizes) + ["Validation"] * len(train_sizes),
    }
).assign(
    ymin=lambda d: d["score"] - d["err"],
    ymax=lambda d: d["score"] + d["err"],
)
(
    ggplot(lc_df, aes(x="n", y="score", color="split", fill="split"))
    + geom_line(size=1.2)
    + geom_ribbon(aes(ymin="ymin", ymax="ymax"), alpha=0.15, color=None)
    + scale_color_manual(values={"Train": INFO, "Validation": WARNING})
    + scale_fill_manual(values={"Train": INFO, "Validation": WARNING})
    + labs(
        title="Learning Curve -- LinearRegression on ADR",
        subtitle="Gap between train and validation = variance; flat high error = bias",
        x="Training Set Size",
        y="MAE (lower = better)",
    )
    + modern_theme(grid=True, legend_pos="bottom")
)

::: {.callout-tip icon=false}
## <i class="bi bi-lightbulb-fill" style="color:#8B5CF6"></i>&nbsp; Pro Tip: read the learning curve

If train and validation scores have **converged** and are both high-error: more data will not help -- the model lacks capacity.

If there is a **large gap** that shrinks as n grows: more data reduces variance. The gap tells you roughly how much.
:::

::: {.callout-tip icon=false}
## <i class="bi bi-puzzle-fill" style="color:#059669"></i>&nbsp; Activity 4: learning curve for LogisticRegression

Compute and plot a learning curve for `clf_pipe` using `scoring='f1'` and `StratifiedKFold`.

```python
# Your code here
```

**Question:** does the classifier need more data or a more complex model?
:::

## 9. Saving and Loading a Model

**From Part 13 (Dev Tools, Reproducibility):** a trained model is an artefact. Save the full bundle -- scaler, model, and feature list -- so downstream code can reconstruct the prediction pipeline exactly.

### 9.1 Save the training bundle

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
model_path: Path = MODEL_DIR / "lin_reg_adr.joblib"
bundle: dict[str, Any] = {
    "scaler": scaler,
    "model": lin_reg,
    "features": FEATURES,
    "target": TARGET_REG,
}
joblib.dump(bundle, model_path)
logger.success(f"Bundle saved to {model_path}")

### 9.2 Reload and predict on one booking

In [ ]:
loaded: dict[str, Any] = joblib.load(model_path)
sample: np.ndarray = X_te[:1]
sample_scaled: np.ndarray = loaded["scaler"].transform(sample)
predicted: float = float(loaded["model"].predict(sample_scaled)[0])
actual: float = float(y_te[0])
logger.info(f"Predicted ADR: {predicted:.2f}  Actual ADR: {actual:.2f}")

::: {.callout-tip icon=false}
## <i class="bi bi-lightbulb-fill" style="color:#8B5CF6"></i>&nbsp; Pro Tip: save the scaler with the model

Save the scaler **inside** the bundle, never separately. A model evaluated on unscaled inputs in production is one of the most common silent bugs in deployed ML systems. The scaler and model must travel together.
:::

## Capstone: End-to-End Training Function

Combine the full workflow -- split, scale, fit, evaluate, save -- into a single typed function.
This is the design pattern you will use in Part 26 and throughout the MLOps path.

### Define the capstone function

In [ ]:
def train_and_evaluate(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    task: str,
    out_dir: Path = Path("models"),
) -> dict[str, Any]:
    """Run the full train-evaluate-save workflow for one task."""
    X_all: np.ndarray = df[features].to_numpy(dtype=float)
    y_all: np.ndarray = df[target].to_numpy()

    stratify: np.ndarray | None = y_all if task == "classification" else None
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=stratify)
    sc: StandardScaler = StandardScaler()
    Xtr_s: np.ndarray = sc.fit_transform(Xtr)
    Xte_s: np.ndarray = sc.transform(Xte)

    if task == "regression":
        model: Any = LinearRegression()
        baseline: Any = DummyRegressor(strategy="mean")
    else:
        model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
        baseline = DummyClassifier(strategy="most_frequent")

    baseline.fit(Xtr_s, ytr)
    model.fit(Xtr_s, ytr)

    if task == "regression":
        m: dict[str, float] = {
            "baseline_mae": float(mean_absolute_error(yte, baseline.predict(Xte_s))),
            "model_mae": float(mean_absolute_error(yte, model.predict(Xte_s))),
            "model_r2": float(r2_score(yte, model.predict(Xte_s))),
        }
    else:
        m = {
            "baseline_f1": float(f1_score(yte, baseline.predict(Xte_s), zero_division=0)),
            "model_f1": float(f1_score(yte, model.predict(Xte_s))),
        }

    out_dir.mkdir(exist_ok=True)
    save_path: Path = out_dir / f"{target}.joblib"
    joblib.dump({"scaler": sc, "model": model, "features": features, "target": target}, save_path)
    logger.success(f"Saved {task} bundle for '{target}' to {save_path}")
    logger.info(f"Metrics: {m}")
    return {"model": model, "scaler": sc, "metrics": m}

### Run the capstone for regression (ADR)

In [ ]:
reg_result: dict[str, Any] = train_and_evaluate(df, FEATURES, TARGET_REG, task="regression", out_dir=MODEL_DIR)

### Run the capstone for classification (cancellation)

In [ ]:
clf_result: dict[str, Any] = train_and_evaluate(df, FEATURES, TARGET_CLF, task="classification", out_dir=MODEL_DIR)

## Further Reading

- [Scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html) -- comprehensive API reference
- Geron, A. (2022), *Hands-On Machine Learning*, Ch 2-4 -- end-to-end pipeline walkthrough
- Antonio et al. (2019), *Hotel booking demand datasets*, Data in Brief -- dataset reference
- [sklearn Pipeline docs](https://scikit-learn.org/stable/modules/compose.html) -- full ColumnTransformer + Pipeline API (covered in Part 26)

**Next up:** Part 26 (nb05) extends this to a full mixed feature set, introduces `ColumnTransformer` and `SimpleImputer`, and covers GridSearchCV, RandomizedSearchCV, and Optuna hyperparameter optimisation.

## 10. Modern sklearn: Pandas Output and Feature Names

Since sklearn 1.2 two changes make pipelines much easier to debug in notebooks:

1. `set_output(transform="pandas")` -- every transformer returns a `DataFrame` with named columns instead of an anonymous numpy array.
2. `get_feature_names_out()` -- every transformer (including `OneHotEncoder`) exposes the name of every output column.

Together they close the gap between sklearn transformers and pandas DataFrames.

### 10.1 Enable pandas output globally

One line affects every transformer in the session. You can also call `.set_output(transform="pandas")` on a single transformer or pipeline if you prefer local control.

In [ ]:
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

set_config(transform_output="pandas")

CATEGORICAL_FEATURES: list[str] = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "customer_type",
    "deposit_type",
]

cat_enc: OneHotEncoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_enc.fit_transform(df[CATEGORICAL_FEATURES]).head(3)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'><span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>set_output</code> vs numpy arrays</span><br><br>Before sklearn 1.2, <code>fit_transform()</code> returned an anonymous <code>ndarray</code>: column indices had no names, making debugging painful. With <code>set_config(transform_output="pandas")</code>, the same call returns a <code>DataFrame</code> where column names come from <code>get_feature_names_out()</code> automatically. The model training code does not change -- sklearn still accepts DataFrames as input.</div>

### 10.2 Inspect feature names after encoding

`get_feature_names_out()` returns the names of every column the transformer produces -- including the expanded OHE columns like `hotel_City Hotel`.

In [ ]:
feature_names: list[str] = cat_enc.get_feature_names_out().tolist()
print(f"OneHotEncoder produced {len(feature_names)} columns:")
for name in feature_names[:10]:
    print(f"  {name}")
print("  ...")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'><span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: inspect a full pipeline's feature names</span><br><br>A multi-step <code>Pipeline</code> exposes the final transformer's output names via <code>pipe[:-1].get_feature_names_out()</code> (slice off the estimator step). For a <code>ColumnTransformer</code>, call <code>ct.get_feature_names_out()</code> directly after fitting -- it concatenates all transformer output names in the order they appear.</div>

### 10.3 Threshold tuning with TunedThresholdClassifierCV

`LogisticRegression` predicts class 1 when `predict_proba >= 0.5` by default. For imbalanced classes -- like hotel cancellations -- the business cost of a false negative (missed cancellation) may differ from a false positive (unnecessary overbooking). `TunedThresholdClassifierCV` (sklearn 1.5) wraps any classifier and finds the optimal decision threshold by cross-validation on a business metric.

In [ ]:
from sklearn.calibration import TunedThresholdClassifierCV

tuned_clf: TunedThresholdClassifierCV = TunedThresholdClassifierCV(
    estimator=clf_pipe,
    scoring="f1",
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    store_cv_results=True,
)
tuned_clf.fit(Xc_tr, yc_tr)
print("Default threshold:  0.500")
print(f"Tuned threshold:    {tuned_clf.best_threshold_:.3f}")
print(f"Best CV F1:         {tuned_clf.best_score_:.3f}")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'><span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: threshold vs regularisation</span><br><br>Changing the decision threshold does not retrain the model -- it shifts the boundary between predicted positives and negatives along the probability axis. This is a post-training calibration step. It is complementary to (not a replacement for) choosing the right regularisation strength, which is a training-time decision. Use <code>TunedThresholdClassifierCV</code> once you are satisfied with the model's AUC; use <code>GridSearchCV</code> when you want to improve AUC itself.</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'><span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity -- Compare thresholds</span><br><br><b>Goal:</b> Predict on <code>Xc_te</code> using both <code>log_reg</code> (default 0.5 threshold) and <code>tuned_clf</code> (tuned threshold). Print a <code>classification_report</code> for each. Which has better recall on class 1 (cancellations)?<br><br><b>Hint:</b> <code>tuned_clf.predict(Xc_te)</code> automatically applies the tuned threshold.</div>

In [ ]:
# TODO: compare classification_report for log_reg vs tuned_clf
...